# VNICT2026 GeoFormerDock — kiem tra khong can GPU + tach validation

Notebook nay chi lam **3 viec khong can GPU that su** (Tier 1/2/3 trong ke hoach
`docs/revision_plan_reviews.md`), truoc khi dung den GPU that cho training:

1. **Tier 1** — kiem tra 5 gia tri `--geo_ablation` moi them vao GeoFormerDock
   (`tools/verify_geo_ablation.py`): dung torch thuan, khong can `data/`.
2. **Tier 2** — smoke test pipeline training that (molgrid + torch + ignite) tren
   2 mau co san trong repo (`demo_inference/`), khong can tai `data/` 80GB.
3. **Tier 3** — tach tap validation tu train theo receptor
   (`tools/make_val_split.py`), chi can file `.types` (vai MB), khong can
   80GB cau truc protein.

**⚠️ TRUOC KHI CHAY:** vao **Settings (panel phai) → Internet → On**. Va **KHONG
bat GPU accelerator** — ca 3 tier o day deu chay CPU, bat GPU chi ton quota
tuan cua ban ma khong dung den.

Neu Tier 1 FAIL: dung lai, dan output vao chat cho Claude, DUNG chay tiep.


## 0. Kiem tra moi truong Kaggle da co san gi

In [ ]:
import sys, subprocess
print('Python:', sys.version)
for pkg in ['numpy', 'torch']:
    try:
        mod = __import__(pkg)
        print(f'{pkg}: {mod.__version__} (da co san)')
    except ImportError:
        print(f'{pkg}: CHUA CO — se can cai them')


## 1. Lay code moi nhat tu GitHub

Repo public: `https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git` — khong can dang nhap/lien ket tai khoan gi, chi can
Internet: On o buoc tren.

In [ ]:
import os
if os.path.isdir('/kaggle/working/VNICT2026_Docking_Paper'):
    print('/kaggle/working/VNICT2026_Docking_Paper da ton tai — pull thay vi clone lai')
    !cd /kaggle/working/VNICT2026_Docking_Paper && git pull
else:
    !cd /kaggle/working && git clone https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git


In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
!git log --oneline -1
print()
print('>>> Dan dong commit tren vao chat de Claude xac nhan ban dang chay ban moi nhat.')


## 2. Tier 1 — kiem tra `--geo_ablation` (KHONG can GPU, KHONG can `data/`)

Chi dung `torch` (Kaggle da cai san). Kiem tra:
1. Ca 5 gia tri `--geo_ablation` deu construct + forward pass duoc.
2. `geo_ablation="none"` cho DUNG 1.594.573 tham so (bang so da bao cao trong bai).
3. Checkpoint cu (neu co trong repo) load duoc vao model "none" moi, khong
   missing/unexpected keys.

**Neu cell duoi FAIL: dung lai, dan toan bo output cho Claude, DUNG chay Tier 2/3.**

In [ ]:
!python3 tools/verify_geo_ablation.py


## 3. Tier 2 — smoke test pipeline training that (chi chay neu Tier 1 PASS)

Can cai them `molgrid` (thu vien tao luoi voxel, khong co san tren Kaggle) va
`pytorch-ignite`, `mlflow`. **`molgrid` yeu cau `numpy<2`** — cell duoi ep phien
ban numpy truoc, co the khien pip cai/ha cap vai goi khac Kaggle co san (binh
thuong, khong phai loi).

Dung `demo_inference/` co san trong repo (4 file `.gninatypes` + 1 file `.types`,
vai chuc KB) — KHONG tai `data/` 80GB. Chay 3 epoch tren 2 mau, mat vai giay.

In [ ]:
!pip install -q 'numpy<2' molgrid pytorch-ignite mlflow


### ⚠️ BAT BUOC: Restart kernel truoc khi chay tiep

`numpy` la mot C-extension — neu cell tren vua doi phien ban numpy,
`importlib.reload()` KHONG dang tin cay de nap lai dung ban moi (day la mot
gotcha quen thuoc cua Jupyter, khong rieng gi notebook nay). Cach chac chan
duy nhat: **Kernel menu → Restart Kernel** (khong chon "Restart & Run All" —
chi restart, roi tu chay tiep tu cell duoi, KHONG chay lai tu dau vi Tier 1
da xong roi).

Sau khi restart, kernel mat toan bo state (ke ca vi tri thu muc) — cell duoi
se tu `cd` lai vao repo va kiem tra numpy truoc khi chay smoke test.

In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
import numpy
print('numpy:', numpy.__version__)
assert numpy.__version__.startswith('1.'), (
    f'numpy={numpy.__version__} van >=2 sau khi cai va restart kernel — '
    'molgrid se import loi. Kiem tra lai cell pip install o tren.'
)
print('OK — numpy < 2, an toan de import molgrid.')


In [ ]:
!bash scripts/run_geoformerdock_ablations.sh smoketest


## 4. Tier 3 — tach validation split (doc lap voi Tier 1/2, khong can GPU/molgrid)

Chi tai file `.types` (vai MB), KHONG tai `PDBbind2016.tar.gz` (80GB cau truc
protein — khong can cho buoc nay).

In [ ]:
!mkdir -p data
!wget -q -O paper_types.tar.gz https://bits.csb.pitt.edu/files/crossdock2020/v1.0/paper_types.tar.gz
!tar -xzf paper_types.tar.gz -C data
!ls -la data/types/


In [ ]:
!python3 tools/make_val_split.py \
    --train data/types/ref_uff_train0.types \
    --out_train data/types/ref_uff_train0_split.types \
    --out_val data/types/ref_uff_val0.types \
    --val_frac 0.15 --seed 2026


**Kiem tra A0b (doc trong `docs/revision_plan_reviews.md`):** trong output cua
cell tren phai co dong `Giao receptor train/val (phai = 0)          : 0`.
Neu con so cuoi khac 0, DUNG lai va bao Claude — co loi ro ri du lieu.

In [ ]:
# Dem so mau y_aff > 0 trong tap val moi tach (A0b: can >= 500 de C-index
# tren val du on dinh de chon checkpoint).
def count_pos(path):
    n_lines = n_pos = 0
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            n_lines += 1
            if float(line.split()[1]) > 0:
                n_pos += 1
    return n_lines, n_pos

for split in ['train0_split', 'val0']:
    n, npos = count_pos(f'data/types/ref_uff_{split}.types')
    print(f'{split:14s}: {n:7d} dong, {npos:5d} mau y_aff>0 ({100*npos/n:.1f}%)')


## 5. Tom tat — dan phan nay vao chat cho Claude

Chay cell duoi roi copy toan bo output gui lai, kem ket qua Tier 1/2/3 o tren.

In [ ]:
import subprocess
print('=== TOM TAT DE GUI LAI ===')
print('git commit:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
print('numpy:', __import__('numpy').__version__)
try:
    print('torch:', __import__('torch').__version__)
except ImportError:
    print('torch: KHONG CO')
try:
    __import__('molgrid')
    print('molgrid: import OK')
except ImportError as e:
    print('molgrid: KHONG import duoc —', e)
import os
for f in ['data/types/ref_uff_train0_split.types', 'data/types/ref_uff_val0.types']:
    print(f, '-> co san' if os.path.exists(f) else '-> THIEU')
